# **Import Dataset**

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("/kaggle/input/datasets/sapampritamjit/datefruit-dataset/DateFruit_Dataset.csv")
df.head()

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [3]:
X = df.drop("Class", axis = 1)
y = df["Class"]

In [4]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42
)

In [6]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# **ANN**

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [8]:
X_train_tensor = torch.tensor(X_train_scaled, dtype = torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype = torch.float32)

y_train_tensor = torch.tensor(y_train, dtype = torch.long)
y_test_tensor = torch.tensor(y_test, dtype = torch.long)

In [9]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [10]:
train_loader = DataLoader(train_dataset, batch_size = 32, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = 32, shuffle = True)

# **Defined Model**

In [11]:
class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()

        self.model = nn.Sequential(
            # 1st hidden layer
            nn.Linear(X.shape[1], 64),
            nn.ReLU(),

            # 2nd hidden layer
            nn.Linear(64, 64),
            nn.ReLU(),

            # output
            nn.Linear(64, 7)
        )


    def forward(self, x):
        return self.model(x)

In [13]:
model = ANN()

# loss & optim

criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

# **Train Model**

In [19]:
epochs = 100
best_val_loss = float("inf")
patience = 5
epoch_without_improvenment = 0

for epoch in range(epochs):
    model.train()

    running_loss = 0.0

    for xb, yb in train_loader:
        optimizer.zero_grad()
        outputs = model(xb)
        loss = criteria(outputs, yb)
        loss.backward()
        optimizer.step() # paramters update

        running_loss += loss

    train_loss = running_loss / len(train_loader)

    # validation
    model.eval()
    running_val_loss = 0.0

    with torch.no_grad():
        for xb, yb in test_loader:
            outputs = model(xb)
            loss = criteria(outputs, yb)
            running_val_loss += loss

    val_loss = running_val_loss / len(test_loader)
    
    print(f"epoch = {epoch + 1}/{epochs} ==> train loss = {train_loss} & val loss = {val_loss}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_model.pt")
        print("Validation loss improves. Model Saved")

    else:
        epoch_without_improvenment += 1
        print(f"No improvenment: {epoch_without_improvenment}/{patience}")

    if epoch_without_improvenment >= patience:
        print("Early stop triggered!")
        break
        

epoch = 1/100 ==> train loss = 1.1966875791549683 & val loss = 0.8859578967094421
Validation loss improves. Model Saved
epoch = 2/100 ==> train loss = 0.746441125869751 & val loss = 0.5839918851852417
Validation loss improves. Model Saved
epoch = 3/100 ==> train loss = 0.5493486523628235 & val loss = 0.4655205309391022
Validation loss improves. Model Saved
epoch = 4/100 ==> train loss = 0.44907814264297485 & val loss = 0.3858075439929962
Validation loss improves. Model Saved
epoch = 5/100 ==> train loss = 0.3889712393283844 & val loss = 0.3569204807281494
Validation loss improves. Model Saved
epoch = 6/100 ==> train loss = 0.342975914478302 & val loss = 0.3185001611709595
Validation loss improves. Model Saved
epoch = 7/100 ==> train loss = 0.3213340640068054 & val loss = 0.2783128023147583
Validation loss improves. Model Saved
epoch = 8/100 ==> train loss = 0.2817677855491638 & val loss = 0.2376304417848587
Validation loss improves. Model Saved
epoch = 9/100 ==> train loss = 0.25693064

# **Evaluation**

In [20]:
model.eval()

total = 0
correct = 0

with torch.no_grad():
    for xb, yb in test_loader:
        outputs = model(xb) # eg :- [0.2, 0.1, 2.2, -1.6, 3.1, 0.8, 1.9] -> 7 values (raw value)
        _, predicted = torch.max(outputs, 1) # return maximum output and its index from eg. value 3.1 and index 4

        correct += (predicted == yb).sum().item()
        total += yb.size(0) # actualk sample in each batch

print(f"total val: {total}, correct val: {correct}")
print("accuracy: ", correct / total * 100)

total val: 180, correct val: 169
accuracy:  93.88888888888889
